# Vanishing-Penny — the teardown 🪙🔬
### The half-life of a risk-free Polymarket gap: episodes, two estimators, the resolution moat

The rigorous companion to [`01_for_the_curious.ipynb`](01_for_the_curious.ipynb). Same seven beats, full method. Thesis: the `YES+NO` arbitrage is **real** (risk-free, ~\$40M documented) but its **half-life is below the public minute tape**, so realised retail capture `½^(latency/H)` ≈ 0 — a mirage that is entirely an execution moat, not a signal failure.

> ⚠️ **Executed on the synthetic book** (baked-in 6-min half-life), where the estimator works — this validates the stopwatch end-to-end. The real verdict is quoted from [`../docs/results.md`](../docs/results.md) (`examples/verify_real.py`). Fixed seeds; no network.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))           # study root (prediction_arb/ lives there)
sys.path.insert(0, os.path.abspath("../../.."))      # repo root, for quantlab
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from prediction_arb import data, arbitrage, robustness

# Offline synthetic book: arbitrage gaps that decay with a KNOWN 6-minute half-life. This
# is where the estimator SHOULD work -- so it validates the machinery. The real verdict
# (the penny closes below the 1-minute tape floor) is in ../docs/results.md via verify_real.py.
gap, truth = data.synthetic_markets(seed=0)
print(f"{gap.shape[1]} markets x {gap.shape[0]} minutes, baked-in half-life = {truth.half_life_min} min")


48 markets x 6000 minutes, baked-in half-life = 6.0 min


## 1 · The claim, as a first-passage hypothesis

H₀ (the thread's hope): `g = 1 − (p_yes + p_no)` persists seconds-to-minutes — long enough to place two CLOB legs and bank `g`.
H₁ (prior): competing bots close `g` far below human reaction, so the **half-life `H`** is the whole game and `½^(latency/H) ≈ 0`.
We test H₀ by *measuring `H`* — first on synthetic data where `H` is known.

In [2]:
eps = arbitrage.detect_all(gap, open_threshold=truth.open_threshold)
print('episodes:', len(eps))
print('summary:', {k:(round(v,3) if isinstance(v,float) else v) for k,v in arbitrage.summary(eps).items()})

episodes: 1647
summary: {'n_episodes': 1647, 'half_life_median_min': 6.0, 'half_life_fit_min': 6.193, 'frac_below_floor': 0.485, 'median_peak_penny': 0.067, 'median_duration_min': 7.0, 'frac_buy_both': 0.512}


## 3–4 · Two estimators must agree (or the decay isn't exponential)

`time_to_half` is assumption-light (median per-episode steps to half its peak); `fit_half_life` is a pooled through-origin slope of `log(|g|/|g_peak|)` on Δt. On a clean exponential they coincide — and on the synthetic they land on the baked-in 6.

In [3]:
print(f"baked-in   : {truth.half_life_min:.2f} min")
print(f"time-to-half: {arbitrage.time_to_half(eps):.2f} min")
print(f"log-lin fit : {arbitrage.fit_half_life(eps):.2f} min")
print('bootstrap CI:', {k:(round(v,2) if isinstance(v,float) else v) for k,v in robustness.bootstrap_half_life(eps, n_boot=2000).items()})

baked-in   : 6.00 min
time-to-half: 6.00 min
log-lin fit : 6.19 min


bootstrap CI: {'half_life_min': 6.0, 'ci_low': 6.0, 'ci_high': 7.0, 'n_episodes': 849, 'n_boot': 2000}


> On **real** data these read differently — and that *is* the result: `half_life_median = nan` because **100% of episodes are below the 1-minute floor** (`frac_below_floor = 1.0`, median duration 1 min). The bootstrap CI is `nan` by construction (no finite per-episode half-lives to resample); the `7.72` log-lin fit survives only on the rare slow tail and is selection-biased upward.

## 4 · The resolution sweep — the load-bearing caveat, as a number

Re-detect on a deliberately coarsened tape. On real data the median episode is *exactly one sample* at **every** fidelity (1→60 min) while the count collapses — the fingerprint of a timescale below all of them. On the synthetic (true `H` = 6 min) the sweep instead inflates past 6 only once the grid is coarser than the half-life:

In [4]:
display(robustness.resolution_sweep(gap).round(3))

,n_episodes,frac_episodes_seen,median_duration_min,half_life_min
fidelity_min,,,,
1,1647,1.0000,7.0000,6.0000
2,1577,0.9570,8.0000,6.0000
5,1397,0.8480,10.0000,10.0000
15,813,0.4940,15.0000,15.0000
30,390,0.2370,30.0000,30.0000
60,193,0.1170,60.0000,60.0000


## 5 · The verdict, with the numbers

**Signal `REAL`** (risk-free; 161 real ≥3¢ gaps, median 4¢; ~\$40M documented). **Tradability `MIRAGE`** (100% sub-floor, median life 1 min; capture 3% at 5-min latency, ~0 at 30). **Execution-moat `CONFIRMED`** (median life = sampling interval at every fidelity; the ~30-ms wallets own it). Real numbers: as-of 2026-06-01, fingerprint `7baea17d9b7b`, in [`../docs/results.md`](../docs/results.md).

## 6 · Could you trade it — the capture integral

Break-even is moot (the gross is risk-free); the binding quantity is the **capture fraction** `½^(latency/H)`. With `H` sub-minute and any human `latency` in minutes, it's rounding error — *before* the two half-spreads and gas the sequential-CLOB fill costs you. This is the cost sweep's analogue: there's no spread at which a minutes-late human captures a seconds-lived penny.

In [5]:
import numpy as np
for H in (0.25, 0.5, 1.0, 6.0):
    row = {f'{lat}m': round(robustness.retail_capture(H, lat),4) for lat in (1,5,30)}
    print(f'H={H:>4} min  capture @', row)

H=0.25 min  capture @ {'1m': 0.0625, '5m': 0.0, '30m': 0.0}
H= 0.5 min  capture @ {'1m': 0.25, '5m': 0.001, '30m': 0.0}
H= 1.0 min  capture @ {'1m': 0.5, '5m': 0.0312, '30m': 0.0}
H= 6.0 min  capture @ {'1m': 0.8909, '5m': 0.5612, '30m': 0.0312}


## 7 · Going further

- **Sub-second `H`** from on-chain `OrderFilled` events (the paper's 86M-tx substrate) — measure the moat instead of bounding it.
- **Combinatorial arbitrage** (the \$29M class): cross-market dependency graph, its own half-lives and the paper's 45% fill rate.
- **Capacity from pre-2026-02-20 depth snapshots** — the professional's constraint this study brackets.

Engine: [`../../../quantlab/`](../../../quantlab/). Method: [`METHODOLOGY.md`](../../../METHODOLOGY.md). Real run: `examples/verify_real.py`.